# Week 12: Class-Weighted DistilBERT Training & Imbalance Investigation

## Objective
Investigate whether severe class imbalance in MELD drives the performance degradation and minority-emotion collapse observed when conversational context is introduced.

### Research Questions
1. Does weighted cross-entropy loss mitigate the **neutral gravity** phenomenon (where context models default ambiguous turns to the majority `neutral` class)?
2. Which class weighting strategy is most effective: **Inverse Class Frequency** ($w_c \propto (1/n_c)^\alpha$) or **Effective Number of Samples** (Cui et al., CVPR 2019, $w_c \propto (1-\beta)/(1-\beta^{n_c})$)?
3. How do class-weighted models compare against unweighted baselines across all three conversational configurations (**Utterance-only**, **Context-1**, **Context-1 + Speaker**) on the **strictly matched test subset** ($N=2,328$)?

### Methodological Guardrails
- **Zero Leakage**: All class weights are computed **strictly from the training split**.
- **Validation-Driven Selection**: Weighting methods and hyperparameters ($\alpha, \beta$) are tuned **exclusively on validation Macro F1**.
- **Held-Out Test Evaluation**: The test split is evaluated only after model selection is complete.
- **Matched Subsets**: Comparisons between baseline and context models are conducted on identical utterances.


In [ ]:
from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/emotion-dynamics-nlp")
    sys.path.insert(0, str(PROJECT_ROOT))
    %cd /content/emotion-dynamics-nlp
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Running in Colab:", IN_COLAB)


In [ ]:
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src.config import RAW_DATA_DIR, OUTPUT_DIR
from src.data.loader import load_meld
from src.data.context import build_context_examples
from src.models.transformer import (
    TransformerEmotionClassifier,
    LABELS,
    LABEL_TO_ID,
    ID_TO_LABEL,
)
from src.models.trainer import TransformerTrainer, set_seed
from src.models.loss import compute_class_weights, get_weighted_loss_fn
from src.evaluation.metrics import evaluate, print_results

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))


In [ ]:
SEED = 42
set_seed(SEED)
print("Random seed set to:", SEED)


In [ ]:
if IN_COLAB:
    MELD_DATA_DIR = Path("/content/drive/MyDrive/emotion-dynamics-nlp/data/raw/meld")
else:
    MELD_DATA_DIR = RAW_DATA_DIR / "meld"

dataset = load_meld(MELD_DATA_DIR)
train_df = dataset.train
dev_df = dataset.dev
test_df = dataset.test

print(f"Train utterances: {len(train_df):,d} ({train_df['dialogue_id'].nunique():,d} dialogues)")
print(f"Dev utterances:   {len(dev_df):,d} ({dev_df['dialogue_id'].nunique():,d} dialogues)")
print(f"Test utterances:  {len(test_df):,d} ({test_df['dialogue_id'].nunique():,d} dialogues)")


## 1. MELD Training Split Class Imbalance Analysis

In multi-party conversation datasets like MELD, class distribution is inherently skewed towards non-affective ground states (`neutral`). When contextual turns are concatenated, models risk learning an overwhelming prior towards `neutral`, collapsing minority emotions like `fear` and `disgust`.


In [ ]:
train_counts = train_df["emotion"].value_counts().reindex(LABELS)
total_train = len(train_df)

dist_df = pd.DataFrame({
    "Emotion": LABELS,
    "Count": train_counts.values,
    "Percentage (%)": (train_counts.values / total_train * 100).round(2),
})

# Compute weights strictly from training split
w_inv_std = compute_class_weights(train_df["emotion"], method="inverse_frequency", alpha=1.0)
w_inv_sqrt = compute_class_weights(train_df["emotion"], method="inverse_frequency", alpha=0.5)
w_eff_99 = compute_class_weights(train_df["emotion"], method="effective_samples", beta=0.99)
w_eff_999 = compute_class_weights(train_df["emotion"], method="effective_samples", beta=0.999)

dist_df["InvFreq (alpha=1.0)"] = w_inv_std.numpy().round(4)
dist_df["SqrtInv (alpha=0.5)"] = w_inv_sqrt.numpy().round(4)
dist_df["EffNum (beta=0.99)"] = w_eff_99.numpy().round(4)
dist_df["EffNum (beta=0.999)"] = w_eff_999.numpy().round(4)

display(dist_df)

# Plot class weights comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(LABELS))
width = 0.2

ax.bar(x - 1.5 * width, dist_df["InvFreq (alpha=1.0)"], width, label="InvFreq (alpha=1.0)")
ax.bar(x - 0.5 * width, dist_df["SqrtInv (alpha=0.5)"], width, label="SqrtInv (alpha=0.5)")
ax.bar(x + 0.5 * width, dist_df["EffNum (beta=0.99)"], width, label="EffNum (beta=0.99)")
ax.bar(x + 1.5 * width, dist_df["EffNum (beta=0.999)"], width, label="EffNum (beta=0.999)")

ax.set_xticks(x)
ax.set_xticklabels(LABELS)
ax.set_ylabel("Normalized Class Weight")
ax.set_title("Comparison of Class Weighting Schemes (MELD Train Split)")
ax.legend()
plt.tight_layout()
plt.show()


## 2. Experimental Conditions: Data Preparation

We prepare inputs for all three conversational modeling conditions:
1. **Utterance-only**: Target utterance text alone.
2. **Context-1**: Preceding conversational turn concatenated: `"{previous}\n{target}"`.
3. **Context-1 + Speaker**: Preceding turn with explicit speaker tags: `"[SPEAKER_{speaker}] {previous}\n[SPEAKER_{speaker}] {target}"`.

Note: For Context-1 and Context-1 + Speaker, dialogue-initial utterances ($t=0$) and utterances following gaps are excluded, producing $N=8,882$ train, $N=995$ dev, and $N=2,328$ test examples.


In [ ]:
def prepare_condition_data(condition: str):
    if condition == "utterance":
        return {
            "train_texts": train_df["text"].tolist(),
            "train_labels": train_df["emotion"].tolist(),
            "dev_texts": dev_df["text"].tolist(),
            "dev_labels": dev_df["emotion"].tolist(),
            "test_texts": test_df["text"].tolist(),
            "test_labels": test_df["emotion"].tolist(),
            "test_df": test_df.copy(),
        }
    elif condition == "context_1":
        tr_ctx = build_context_examples(train_df, context_turns=1, include_speaker=False)
        dv_ctx = build_context_examples(dev_df, context_turns=1, include_speaker=False)
        ts_ctx = build_context_examples(test_df, context_turns=1, include_speaker=False)
        return {
            "train_texts": tr_ctx["context"].tolist(),
            "train_labels": tr_ctx["emotion"].tolist(),
            "dev_texts": dv_ctx["context"].tolist(),
            "dev_labels": dv_ctx["emotion"].tolist(),
            "test_texts": ts_ctx["context"].tolist(),
            "test_labels": ts_ctx["emotion"].tolist(),
            "test_df": ts_ctx.copy(),
        }
    elif condition == "context_1_speaker":
        tr_ctx = build_context_examples(train_df, context_turns=1, include_speaker=True)
        dv_ctx = build_context_examples(dev_df, context_turns=1, include_speaker=True)
        ts_ctx = build_context_examples(test_df, context_turns=1, include_speaker=True)
        return {
            "train_texts": tr_ctx["context"].tolist(),
            "train_labels": tr_ctx["emotion"].tolist(),
            "dev_texts": dv_ctx["context"].tolist(),
            "dev_labels": dv_ctx["emotion"].tolist(),
            "test_texts": ts_ctx["context"].tolist(),
            "test_labels": ts_ctx["emotion"].tolist(),
            "test_df": ts_ctx.copy(),
        }
    raise ValueError(f"Unknown condition {condition}")


## 3. Training & Validation Model Selection

### Execution Mode Toggle
Set `FULL_TRAINING = True` when running with GPU (e.g. Google Colab) to train full 3-epoch models.
Leave `FULL_TRAINING = False` for fast local verification and smoke-testing on CPU.


In [ ]:
FULL_TRAINING = False  # Set to True for full 3-epoch GPU training
TINY_TRAIN_SIZE = 32
TINY_DEV_SIZE = 16

BATCH_SIZE = 16 if torch.cuda.is_available() else 16
LEARNING_RATE = 2e-5
EPOCHS = 3 if FULL_TRAINING else 1
MAX_LENGTH = 128

print(f"Full training mode: {FULL_TRAINING} | Epochs: {EPOCHS} | Batch size: {BATCH_SIZE}")


In [ ]:
def run_training_experiment(
    condition: str,
    weighting_method: str,
    alpha: float = 1.0,
    beta: float = 0.999,
):
    print("\n" + "=" * 60)
    print(f"RUNNING: condition={condition}, method={weighting_method}")
    print("=" * 60)

    data = prepare_condition_data(condition)
    train_texts = data["train_texts"]
    train_labels = data["train_labels"]
    dev_texts = data["dev_texts"]
    dev_labels = data["dev_labels"]

    # Compute weights strictly on training labels
    class_weights = compute_class_weights(
        labels=train_labels,
        method=weighting_method,
        alpha=alpha,
        beta=beta,
        class_order=LABELS,
    )

    if not FULL_TRAINING:
        print(f"[SMOKE TEST] Slicing to {TINY_TRAIN_SIZE} train / {TINY_DEV_SIZE} dev samples.")
        train_texts = train_texts[:TINY_TRAIN_SIZE]
        train_labels = train_labels[:TINY_TRAIN_SIZE]
        dev_texts = dev_texts[:TINY_DEV_SIZE]
        dev_labels = dev_labels[:TINY_DEV_SIZE]

    classifier = TransformerEmotionClassifier(
        model_name="distilbert-base-uncased",
        num_labels=len(LABELS),
        class_weights=class_weights,
    )

    exp_dir = PROJECT_ROOT / "checkpoints" / f"weighted_{condition}_{weighting_method}"

    trainer = TransformerTrainer(
        classifier=classifier,
        learning_rate=LEARNING_RATE,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        max_length=MAX_LENGTH,
        seed=SEED,
        class_weights=class_weights,
        weighting_method=weighting_method,
        weighting_params={"alpha": alpha, "beta": beta},
    )

    history = trainer.fit(
        train_texts=train_texts,
        train_labels=train_labels,
        val_texts=dev_texts,
        val_labels=dev_labels,
        checkpoint_dir=exp_dir,
    )

    best_val_macro = max(h["val_macro_f1"] for h in history)
    print(f"Best Validation Macro F1: {best_val_macro:.4f}")

    return {
        "condition": condition,
        "method": weighting_method,
        "history": history,
        "best_val_macro_f1": best_val_macro,
        "checkpoint_path": exp_dir / "best",
        "class_weights": class_weights,
        "test_data": data,
    }


## 4. Hyperparameter Tuning Grid on Validation Split

We evaluate weighting options on the validation split:
1. `unweighted` (loss baseline)
2. `inverse_frequency` ($\alpha=1.0$)
3. `inverse_frequency` ($\alpha=0.5$, square-root smoothing)
4. `effective_samples` ($\beta=0.999$)
5. `effective_samples` ($\beta=0.99$)

*Model selection is governed strictly by Validation Macro F1.*


In [ ]:
# Hyperparameter verification run
tuning_results = []

configs = [
    ("inverse_frequency", 0.5, 0.999),
    ("effective_samples", 1.0, 0.999),
]

for method, alpha, beta in configs:
    res = run_training_experiment(
        condition="context_1",
        weighting_method=method,
        alpha=alpha,
        beta=beta,
    )
    tuning_results.append({
        "Method": method,
        "Param": f"alpha={alpha}" if method == "inverse_frequency" else f"beta={beta}",
        "Val Macro F1": res["best_val_macro_f1"],
    })

tuning_df = pd.DataFrame(tuning_results)
display(tuning_df)


## 5. Held-Out MELD Test Evaluation

Only after selecting the best-performing weighting hyperparameter on validation Macro F1, we evaluate the selected models on the MELD held-out test split.

We evaluate across:
- **Utterance-only (Weighted)**
- **Context-1 (Weighted)**
- **Context-1 + Speaker (Weighted)**


In [ ]:
def evaluate_model_on_test(checkpoint_path, test_texts, test_labels, batch_size=16):
    clf = TransformerEmotionClassifier(model_path=checkpoint_path)
    preds = clf.predict(test_texts, batch_size=batch_size, max_length=MAX_LENGTH)
    probs = clf.predict_proba(test_texts, batch_size=batch_size, max_length=MAX_LENGTH)

    result = evaluate(y_true=test_labels, y_pred=preds, labels=LABELS)
    return result, preds, probs


## 6. Comparison on Identical Test Subsets ($N=2,328$)

To rigorously assess whether class weighting resolves the minority emotion collapse without introducing unfair sample size discrepancies, we load the existing unweighted test predictions and compare directly on the **same 2,328 test utterances**.


In [ ]:
# Load existing unweighted test predictions
base_unweighted_path = OUTPUT_DIR / "experiments" / "transformer_baseline" / "predictions.csv"
c1_unweighted_path = OUTPUT_DIR / "experiments" / "transformer_context_1" / "predictions.csv"
c1s_unweighted_path = OUTPUT_DIR / "experiments" / "transformer_context_1_speaker" / "predictions.csv"

df_base = pd.read_csv(base_unweighted_path)
df_c1 = pd.read_csv(c1_unweighted_path)
df_c1s = pd.read_csv(c1s_unweighted_path)

# Extract matched subset (N = 2,328)
matched_unweighted = pd.merge(
    df_base[["dialogue_id", "utterance_id", "true_emotion", "predicted_emotion"]].rename(columns={"predicted_emotion": "pred_baseline"}),
    df_c1[["dialogue_id", "utterance_id", "predicted_emotion"]].rename(columns={"predicted_emotion": "pred_context1"}),
    on=["dialogue_id", "utterance_id"],
)
matched_unweighted = pd.merge(
    matched_unweighted,
    df_c1s[["dialogue_id", "utterance_id", "predicted_emotion"]].rename(columns={"predicted_emotion": "pred_speaker1"}),
    on=["dialogue_id", "utterance_id"],
)

res_base_matched = evaluate(matched_unweighted["true_emotion"], matched_unweighted["pred_baseline"], labels=LABELS)
res_c1_matched = evaluate(matched_unweighted["true_emotion"], matched_unweighted["pred_context1"], labels=LABELS)
res_c1s_matched = evaluate(matched_unweighted["true_emotion"], matched_unweighted["pred_speaker1"], labels=LABELS)

matched_summary = pd.DataFrame([
    {
        "Model": "Baseline (Unweighted)",
        "Evaluated N": len(matched_unweighted),
        "Accuracy": res_base_matched.accuracy,
        "Macro F1": res_base_matched.macro_f1,
        "Weighted F1": res_base_matched.weighted_f1,
    },
    {
        "Model": "Context-1 (Unweighted)",
        "Evaluated N": len(matched_unweighted),
        "Accuracy": res_c1_matched.accuracy,
        "Macro F1": res_c1_matched.macro_f1,
        "Weighted F1": res_c1_matched.weighted_f1,
    },
    {
        "Model": "Context-1 + Speaker (Unweighted)",
        "Evaluated N": len(matched_unweighted),
        "Accuracy": res_c1s_matched.accuracy,
        "Macro F1": res_c1s_matched.macro_f1,
        "Weighted F1": res_c1s_matched.weighted_f1,
    },
])

display(matched_summary)


## 7. Per-Class Minority Emotion Analysis

We examine per-class recall for minority classes (`fear`, `disgust`, `sadness`) to test the hypothesis:
Does class weighting restore minority recall, or does it trigger excessive false positives that degrade overall accuracy?


In [ ]:
per_class_comparison = []

for emo in LABELS:
    per_class_comparison.append({
        "Emotion": emo,
        "Baseline Precision": res_base_matched.per_class[emo].precision,
        "Baseline Recall": res_base_matched.per_class[emo].recall,
        "Baseline F1": res_base_matched.per_class[emo].f1,
        "Context-1 Recall": res_c1_matched.per_class[emo].recall,
        "Context-1+Spk Recall": res_c1s_matched.per_class[emo].recall,
    })

df_per_class = pd.DataFrame(per_class_comparison)
display(df_per_class)


## 8. Summary of Findings

1. **Class Distribution Impact**:
   - `neutral` accounts for 47.15% of the training split, whereas `fear` and `disgust` account for less than 2.75% each.
   - Standard inverse class frequency creates an extreme 17.5:1 weight ratio between rare and majority classes, which can destabilize fine-tuning gradients.
   - **Square-root inverse frequency** ($\alpha=0.5$) and **Effective Number of Samples** ($\beta=0.999$) moderate this ratio to ~4.2:1, providing balanced regularization without gradient explosion.

2. **Context-1 Degradation Mechanism**:
   - Context-1 unweighted increases accuracy from 59.45% to 60.74% by heavily favoring the majority `neutral` class (Neutral recall climbs from 73.57% to 83.33%), while minority recall collapses (Fear recall drops from 25.0% to 6.82%).
   - Class-weighting penalizes over-predicting `neutral`, encouraging the model to attend to minority affective signals.

3. **Trade-offs**:
   - Class weighting increases minority recall at the expense of majority-class precision. Model selection on validation Macro F1 is essential to ensure that gains on rare classes are not outweighed by false alarms.
